# Lab 9: Red neuronal desde cero

Construimos una red neuronal multicapa (MLP) sin usar frameworks de deep learning.
Implementamos propagación hacia adelante, propagación hacia atrás y actualización de parámetros.
El ejemplo de clasificación usa datos de `make_classification` en lugar del dataset H5 de gatos
(que requiere archivos externos), pero la arquitectura y el código son idénticos.

Topología configurable: `Topology = [n_x, n_h1, n_h2, ..., n_y]`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 1. Funciones de activación

Definimos las dos que vamos a usar: `relu` para las capas ocultas y `sigmoid` para la capa de salida (clasificación binaria).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_d(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_d(z):
    return (z > 0).astype(float)

# Mapeo para uso dinámico por nombre
ACTIVACIONES = {
    'relu':    (relu,    relu_d),
    'sigmoid': (sigmoid, sigmoid_d),
}

## 2. Clase RedNeuronal

La clase toma una lista `topology` con el número de neuronas en cada capa y una lista `activaciones`
que especifica la función de activación de cada capa (excepto la de entrada).

Para cada capa $l$ guardamos:
- $W^{[l]}$ de dimensión $(n^{[l]}, n^{[l-1]})$
- $b^{[l]}$ de dimensión $(n^{[l]}, 1)$

y durante el forward pass guardamos $Z^{[l]}$ y $A^{[l]}$ para usarlos en el backward.

In [ ]:
class RedNeuronal:
    """
    Red neuronal feedforward con topología configurable.

    topology:    [n_entrada, n_oculta1, ..., n_salida]
    activaciones: lista de strings ('relu', 'sigmoid') de largo len(topology)-1
    """

    def __init__(self, topology, activaciones):
        assert len(activaciones) == len(topology) - 1
        self.topology = topology
        self.activaciones = activaciones
        self.L = len(topology) - 1  # número de capas con parámetros
        self.params = {}
        self.costo_hist = []
        self._inicializar_parametros()

    def _inicializar_parametros(self):
        for l in range(1, self.L + 1):
            n_l = self.topology[l]
            n_l_prev = self.topology[l - 1]
            # Inicialización He para ReLU, Xavier para sigmoid
            if self.activaciones[l - 1] == 'relu':
                escala = np.sqrt(2.0 / n_l_prev)
            else:
                escala = np.sqrt(1.0 / n_l_prev)
            self.params[f'W{l}'] = np.random.randn(n_l, n_l_prev) * escala
            self.params[f'b{l}'] = np.zeros((n_l, 1))

    def _forward(self, X):
        """
        X: (n_features, m_samples)
        Devuelve el output A[L] y guarda el caché intermedio para backprop.
        """
        cache = {'A0': X}
        A = X

        for l in range(1, self.L + 1):
            W = self.params[f'W{l}']
            b = self.params[f'b{l}']
            Z = W.dot(A) + b
            act_fn, _ = ACTIVACIONES[self.activaciones[l - 1]]
            A = act_fn(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A

        return A, cache

    def _costo_binario(self, AL, y):
        m = y.shape[1]
        eps = 1e-9
        return -(1 / m) * np.sum(y * np.log(AL + eps) + (1 - y) * np.log(1 - AL + eps))

    def _backward(self, cache, y):
        """
        Backpropagation. Devuelve gradientes para todos los parámetros.
        """
        grads = {}
        m = y.shape[1]
        AL = cache[f'A{self.L}']

        # Gradiente de la capa de salida (sigmoid + cross-entropy se simplifica)
        dA = -(y / (AL + 1e-9) - (1 - y) / (1 - AL + 1e-9))

        for l in reversed(range(1, self.L + 1)):
            Z = cache[f'Z{l}']
            A_prev = cache[f'A{l-1}']
            W = self.params[f'W{l}']
            _, act_d = ACTIVACIONES[self.activaciones[l - 1]]

            dZ = dA * act_d(Z)
            grads[f'dW{l}'] = (1 / m) * dZ.dot(A_prev.T)
            grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)
            dA = W.T.dot(dZ)

        return grads

    def fit(self, X, y, alpha=0.01, n_iter=1000, verbose=True):
        """
        X: (m, n_features), y: (m,) con valores 0/1
        """
        # Transponemos para tener (n_features, m) durante el entrenamiento
        Xt = X.T
        yt = y.reshape(1, -1)

        for i in range(n_iter):
            AL, cache = self._forward(Xt)
            costo = self._costo_binario(AL, yt)
            self.costo_hist.append(costo)

            grads = self._backward(cache, yt)

            for l in range(1, self.L + 1):
                self.params[f'W{l}'] -= alpha * grads[f'dW{l}']
                self.params[f'b{l}'] -= alpha * grads[f'db{l}']

            if verbose and (i % 200 == 0 or i == n_iter - 1):
                print(f'  iter {i:>4}: costo = {costo:.6f}')

        return self

    def predict_proba(self, X):
        AL, _ = self._forward(X.T)
        return AL.squeeze()

    def predict(self, X, umbral=0.5):
        return (self.predict_proba(X) >= umbral).astype(int)

## 3. Verificación de dimensiones

Antes de entrenar, comprobamos que el forward pass produce las dimensiones correctas con datos de prueba.

In [ ]:
# Verificación rápida de dimensiones con datos ficticios
topo_prueba = [4, 5, 3, 1]
acts_prueba = ['relu', 'relu', 'sigmoid']
red_prueba = RedNeuronal(topo_prueba, acts_prueba)

X_dummy = np.random.randn(10, 4)  # 10 muestras, 4 features
y_dummy = np.random.randint(0, 2, 10)

prob = red_prueba.predict_proba(X_dummy)
print(f'Input shape:  {X_dummy.shape}')
print(f'Output shape: {prob.shape}  (esperado: (10,))')
print(f'Rango output: [{prob.min():.3f}, {prob.max():.3f}]  (esperado en (0,1))')

for l in range(1, len(topo_prueba)):
    W = red_prueba.params[f'W{l}']
    b = red_prueba.params[f'b{l}']
    print(f'  Capa {l}: W={W.shape}, b={b.shape}')

## 4. Entrenamiento en datos reales

Usamos un dataset de clasificación binaria en 10 features, una capa oculta de 8 neuronas.

In [ ]:
X_data, y_data = make_classification(
    n_samples=800, n_features=10, n_informative=6,
    n_redundant=2, random_state=0
)

X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data, test_size=0.2, random_state=0, stratify=y_data)

# Topología: 10 entradas -> 8 ocultas -> 4 ocultas -> 1 salida
red = RedNeuronal(
    topology=[10, 8, 4, 1],
    activaciones=['relu', 'relu', 'sigmoid']
)

print('Entrenando...')
red.fit(X_tr, y_tr, alpha=0.05, n_iter=1000)

In [ ]:
# Curva de costo
plt.figure(figsize=(8, 4))
plt.plot(red.costo_hist)
plt.xlabel('Iteración')
plt.ylabel('Costo (cross-entropy)')
plt.title('Convergencia del entrenamiento')
plt.grid(alpha=0.25)
plt.show()

acc_tr = accuracy_score(y_tr, red.predict(X_tr))
acc_te = accuracy_score(y_te, red.predict(X_te))
print(f'Accuracy train: {acc_tr:.4f}')
print(f'Accuracy test:  {acc_te:.4f}')

## 5. Efecto de la topología

Comparamos tres arquitecturas distintas para ver cómo la profundidad y el número de neuronas afectan el accuracy.

In [ ]:
arquitecturas = [
    ([10, 4, 1],         ['relu', 'sigmoid'],        'Red pequeña (10-4-1)'),
    ([10, 8, 4, 1],      ['relu', 'relu', 'sigmoid'], 'Red media  (10-8-4-1)'),
    ([10, 16, 8, 4, 1],  ['relu', 'relu', 'relu', 'sigmoid'], 'Red grande (10-16-8-4-1)'),
]

resultados = []
for topo, acts, nombre in arquitecturas:
    r = RedNeuronal(topo, acts)
    r.fit(X_tr, y_tr, alpha=0.05, n_iter=1000, verbose=False)
    a_tr = accuracy_score(y_tr, r.predict(X_tr))
    a_te = accuracy_score(y_te, r.predict(X_te))
    resultados.append({'Arquitectura': nombre, 'Acc Train': a_tr, 'Acc Test': a_te})
    print(f'{nombre}: train={a_tr:.4f}, test={a_te:.4f}')

import pandas as pd
display(pd.DataFrame(resultados))

## Conclusiones

- La implementación desde cero confirma que backpropagation es simplemente la regla de la cadena aplicada a cada capa en orden inverso.
- La inicialización de pesos importa: He para ReLU, Xavier para sigmoid evitan el problema de gradientes que explotan o desaparecen.
- Con topología `[10, 8, 4, 1]` y 1000 iteraciones ya se logra un accuracy razonable.
- Para producción se usaría Keras o PyTorch, que tienen todas estas piezas optimizadas y con más opciones.